# Day 4: Generation Algorithms Deep Dive
Objective: Understand and compare Greedy, Beam Search, and Sampling algorithms.
Reference: https://huggingface.co/blog/how-to-generate

In [9]:
!pip install transformers torch -q

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

prompt = "Artificial intelligence will change"
inputs = tokenizer(prompt, return_tensors="pt")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 1. Greedy Decoding
Always picks the single highest probability token at each step.
- Deterministic: same input = same output every time
- Fast but can get repetitive

In [11]:
output = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=50,
    do_sample=False  # greedy
)
greedy_output = tokenizer.decode(output[0], skip_special_tokens=True)
print("GREEDY:")
print(greedy_output)

GREEDY:
Artificial intelligence will change the way we think about the world.












































## 2. Beam Search
Instead of picking 1 best token at each step, keeps top N sequences (beams) alive.
At the end, returns the sequence with the highest overall score.
- num_beams=5 means it tracks 5 candidate sequences simultaneously
- More coherent than greedy but slower
- Can still be repetitive on long outputs

In [12]:
output = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=50,
    num_beams=5,
    early_stopping=True,
    no_repeat_ngram_size=2  # prevents repeating 2-word phrases
)
beam_output = tokenizer.decode(output[0], skip_special_tokens=True)
print("BEAM SEARCH (num_beams=5):")
print(beam_output)

BEAM SEARCH (num_beams=5):
Artificial intelligence will change the way we think about the world.


## 3. Sampling (temperature)
Instead of always picking the top token, samples randomly based on probabilities.
- temperature < 1.0 = more focused, less creative
- temperature > 1.0 = more creative, less coherent
- temperature = 1.0 = use raw model probabilities

In [13]:
torch.manual_seed(42)  # for reproducibility

output = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
)
temp_output = tokenizer.decode(output[0], skip_special_tokens=True)
print("SAMPLING (temperature=0.7):")
print(temp_output)

SAMPLING (temperature=0.7):
Artificial intelligence will change the landscape of work, and in this case, it will lead to a new and more efficient way of interacting with people.




The technology, which is already being made available in other fields, is already being developed in the field


## 4. Top-K Sampling
Only sample from the top K most probable tokens.
Cuts off the long tail of unlikely tokens completely.
- top_k=50 means only consider the 50 highest probability tokens
- Prevents very unlikely/weird tokens from being picked

In [14]:
torch.manual_seed(42)

output = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    top_k=50
)
topk_output = tokenizer.decode(output[0], skip_special_tokens=True)
print("TOP-K SAMPLING (k=50):")
print(topk_output)

TOP-K SAMPLING (k=50):
Artificial intelligence will change the landscape of work, and in this case, it will lead to a new and more efficient way of interacting with people.




The technology, which is already being made available in other fields, is already being developed in the field


## 5. Top-P (Nucleus) Sampling
Instead of fixed K tokens, dynamically picks the smallest set of tokens
whose probabilities add up to P.
- top_p=0.9 means: keep adding tokens until their combined probability = 90%
- Adapts to context: sometimes 5 tokens cover 90%, sometimes 50 do
- Generally preferred over top_k in practice

In [15]:
torch.manual_seed(42)

output = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    top_k=0,    # disable top_k
    top_p=0.9   # only nucleus sampling
)
topp_output = tokenizer.decode(output[0], skip_special_tokens=True)
print("TOP-P SAMPLING (p=0.9):")
print(topp_output)

TOP-P SAMPLING (p=0.9):
Artificial intelligence will change the way we work, and in this case, it will help us make better decisions about what is best for our country.




The American Academy of Sciences will soon be awarding a grant to the world's first artificial intelligence expert to


## 6. Combined: Top-K + Top-P (Best Practice)
Use both together — industry standard for most LLM applications.
top_k first narrows the pool, top_p then refines it further.

In [16]:
torch.manual_seed(42)

output = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.9
)
combined_output = tokenizer.decode(output[0], skip_special_tokens=True)
print("COMBINED top_k=50 + top_p=0.9:")
print(combined_output)

COMBINED top_k=50 + top_p=0.9:
Artificial intelligence will change the way we work, and in this case, it will help us make better decisions about what is best for our country.




The American Academy of Sciences has just announced that it is funding a $1.3 billion study on


In [17]:
print("=" * 60)
print("PROMPT:", prompt)
print("=" * 60)
print("\n1. GREEDY:")
print(greedy_output)
print("\n2. BEAM SEARCH:")
print(beam_output)
print("\n3. TEMPERATURE SAMPLING:")
print(temp_output)
print("\n4. TOP-K:")
print(topk_output)
print("\n5. TOP-P:")
print(topp_output)
print("\n6. COMBINED:")
print(combined_output)
print("=" * 60)

PROMPT: Artificial intelligence will change

1. GREEDY:
Artificial intelligence will change the way we think about the world.











































2. BEAM SEARCH:
Artificial intelligence will change the way we think about the world.

3. TEMPERATURE SAMPLING:
Artificial intelligence will change the landscape of work, and in this case, it will lead to a new and more efficient way of interacting with people.




The technology, which is already being made available in other fields, is already being developed in the field

4. TOP-K:
Artificial intelligence will change the landscape of work, and in this case, it will lead to a new and more efficient way of interacting with people.




The technology, which is already being made available in other fields, is already being developed in the field

5. TOP-P:
Artificial intelligence will change the way we work, and in this case, it will help us make better decisions about what is best for our country.




The American Acade

## 7. Observations & Notes

### Greedy vs Beam Search
- Greedy picks 1 best token per step → fast but myopic
- Beam search keeps N best sequences → slower but more globally optimal
- Both are deterministic (no randomness)

### Why sampling over greedy/beam?
- Greedy and beam can loop/repeat on open-ended generation
- Sampling introduces controlled randomness → more natural text
- Real LLM APIs (ChatGPT, Claude) use sampling not beam search

### Top-K vs Top-P
- Top-K: fixed number of candidates (always 50)
- Top-P: dynamic number based on probability mass
- Top-P is generally better because it adapts to context
- When model is confident → fewer tokens needed to reach 90%
- When model is uncertain → more tokens included automatically

### Industry standard settings
temperature=0.7, top_k=50, top_p=0.9 — what most production
systems use as a starting point.

### What I still need to understand better
- (write your own questions here after running the cells)